In [ ]:
import os
import sys
import joblib
import pandas as pd
import numpy as np

sys.path.append("../src")
sys.path.append("/workspaces/Phoenix/ai-ml/training_pipeline")

from models.isolation_forest import AI012IsolationForest
from src import TrainingPipeline

In [ ]:
AI012_ROOT = "/workspaces/Phoenix/ai-ml/models/ai012-anomaly"
TRAINING_PIPELINE_ROOT = "/workspaces/Phoenix/ai-ml/training_pipeline"

DATASET_PATH = os.path.join(
    AI012_ROOT,
    "data",
    "raw",
    "anomaly_detection_hourly_2020_2024.csv"
)

LABEL_PATH = os.path.join(
    AI012_ROOT,
    "src",
    "labeling",
    "data",
    "processed",
    "anomaly_labels_v1.csv"
)

CHECKPOINT_PATH = os.path.join(
    AI012_ROOT,
    "checkpoints",
    "isolation_forest.pkl"
)

OUTPUT_DIR = os.path.join(AI012_ROOT, "data", "outputs")

CONFIG_PATH = os.path.join(
    TRAINING_PIPELINE_ROOT,
    "configs",
    "default_config.yaml"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

In [ ]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

if os.path.exists(LABEL_PATH):
    labels_df = pd.read_csv(LABEL_PATH)

    if "row_id" in df.columns and "row_id" in labels_df.columns:
        df = df.merge(
            labels_df[["row_id", "anomaly_label"]],
            on="row_id",
            how="left"
        )
    elif len(df) == len(labels_df):
        df["anomaly_label"] = labels_df["anomaly_label"].values
    else:
        print("Role C labels do not align with this full dataset. Creating evaluation labels from dataset signals.")
        labels_df = None
else:
    labels_df = None

if "anomaly_label" not in df.columns:
    numeric_df = df.select_dtypes(include=[np.number]).copy()
    numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
    numeric_df = numeric_df.fillna(numeric_df.median(numeric_only=True))

    candidate_cols = [
        c for c in numeric_df.columns
        if any(k in c.lower() for k in [
            "risk", "score", "severity", "incident",
            "cyber", "threat", "anomaly", "zscore", "intensity"
        ])
    ]

    if len(candidate_cols) == 0:
        candidate_cols = list(numeric_df.columns)

    combined_score = numeric_df[candidate_cols].rank(pct=True).mean(axis=1)

    threshold = combined_score.quantile(0.95)
    df["anomaly_label"] = (combined_score >= threshold).astype(int)

df["anomaly_label"] = df["anomaly_label"].fillna(0).astype(int)

print(df.shape)
print(df["anomaly_label"].value_counts())

In [ ]:
model = AI012IsolationForest(
    contamination=0.05,
    random_state=42
)

pipeline = TrainingPipeline(
    config_path=CONFIG_PATH,
    root_dir=TRAINING_PIPELINE_ROOT
)

pipeline.set_dataset_frame(
    df,
    target_column="anomaly_label",
    dataset_name="ai012_anomaly_detection_hourly"
)

pipeline.set_model_instance(
    model,
    model_name="ai012_isolation_forest",
    task_type="classification"
)

pipeline.set_training(
    epochs=1,
    batch_size=64,
    verbose=True
)

pipeline.set_output(
    path="checkpoints",
    log_path="logs",
    reports_path="reports",
    checkpoint_prefix="ai012_isolation_forest",
    previous_checkpoints_to_keep=2
)

pipeline.enable_tensorboard(
    True,
    log_dir="logs/tensorboard"
)

result = pipeline.run(
    run_id="ai012_isolation_forest"
)

result

In [ ]:
model.fit(df)

output_df = model.build_output(df)

scores_path = os.path.join(OUTPUT_DIR, "isolation_forest_scores.csv")
flags_path = os.path.join(OUTPUT_DIR, "isolation_forest_flags.csv")

output_df.to_csv(scores_path, index=False)

flag_cols = []

for col in ["event_id", "timestamp", "region", "location", "hazard_type", "threat_type"]:
    if col in output_df.columns:
        flag_cols.append(col)

flag_cols += ["anomaly_flag", "anomaly_score", "anomaly_rank"]

output_df[flag_cols].to_csv(flags_path, index=False)

joblib.dump(model, CHECKPOINT_PATH)

print("Checkpoint:", CHECKPOINT_PATH)
print("Scores:", scores_path)
print("Flags:", flags_path)

output_df.head()

In [ ]:
# Create AI012 required outputs using trained Isolation Forest model

model.fit(df)

output_df = model.build_output(df)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

scores_path = os.path.join(OUTPUT_DIR, "isolation_forest_scores.csv")
flags_path = os.path.join(OUTPUT_DIR, "isolation_forest_flags.csv")

output_df.to_csv(scores_path, index=False)

flag_columns = []

for col in ["event_id", "timestamp", "region", "location", "hazard_type", "threat_type"]:
    if col in output_df.columns:
        flag_columns.append(col)

flag_columns += ["anomaly_flag", "anomaly_score", "anomaly_rank"]

output_df[flag_columns].to_csv(flags_path, index=False)

joblib.dump(model, CHECKPOINT_PATH)

print("Isolation Forest completed.")
print("Rows processed:", len(output_df))
print("Anomalies detected:", int(output_df["anomaly_flag"].sum()))
print("Anomaly rate:", round(float(output_df["anomaly_flag"].mean()), 4))
print("Scores:", scores_path)
print("Flags:", flags_path)
print("Checkpoint:", CHECKPOINT_PATH)

output_df.head()

In [ ]:
# Contamination tuning from 0.01 to 0.10

contamination_values = [0.01, 0.03, 0.05, 0.07, 0.10]
tuning_results = []

for contamination in contamination_values:
    tuning_model = AI012IsolationForest(
        contamination=contamination,
        random_state=42
    )

    tuning_model.fit(df)
    tuning_output = tuning_model.build_output(df)

    anomaly_count = int(tuning_output["anomaly_flag"].sum())
    anomaly_rate = float(tuning_output["anomaly_flag"].mean())

    tuning_results.append({
        "contamination": contamination,
        "rows": len(tuning_output),
        "anomaly_count": anomaly_count,
        "anomaly_rate": anomaly_rate,
        "mean_anomaly_score": float(tuning_output["anomaly_score"].mean()),
        "min_anomaly_score": float(tuning_output["anomaly_score"].min()),
        "max_anomaly_score": float(tuning_output["anomaly_score"].max())
    })

tuning_df = pd.DataFrame(tuning_results)

tuning_path = os.path.join(
    OUTPUT_DIR,
    "isolation_forest_contamination_tuning.csv"
)

tuning_df.to_csv(tuning_path, index=False)

print("Contamination tuning completed.")
print("Tuning results saved to:", tuning_path)

tuning_df

In [ ]:
# Create TensorBoard event logs for AI012 Isolation Forest

from tensorboard.summary.writer.event_file_writer import EventFileWriter
from tensorboard.compat.proto.event_pb2 import Event
from tensorboard.compat.proto.summary_pb2 import Summary
import time

TENSORBOARD_RUN_DIR = "/workspaces/Phoenix/ai-ml/training_pipeline/logs/tensorboard/ai012_isolation_forest"

os.makedirs(TENSORBOARD_RUN_DIR, exist_ok=True)

writer = EventFileWriter(TENSORBOARD_RUN_DIR)

for idx, row in tuning_df.iterrows():
    step = idx + 1

    metrics = {
        "ai012/anomaly_rate": row["anomaly_rate"],
        "ai012/anomaly_count": row["anomaly_count"],
        "ai012/mean_anomaly_score": row["mean_anomaly_score"],
        "ai012/min_anomaly_score": row["min_anomaly_score"],
        "ai012/max_anomaly_score": row["max_anomaly_score"],
    }

    for tag, value in metrics.items():
        summary = Summary(
            value=[
                Summary.Value(
                    tag=tag,
                    simple_value=float(value)
                )
            ]
        )

        event = Event(
            wall_time=time.time(),
            step=step,
            summary=summary
        )

        writer.add_event(event)

writer.close()

print("TensorBoard event logs created.")
print("Log directory:", TENSORBOARD_RUN_DIR)